### Import Modules

In [3]:
import pandas as pd
import numpy as np
import csv



### Rotten Tomatoes Movies & Critic Reviews Dataset  
https://www.kaggle.com/datasets/stefanoleone992/rotten-tomatoes-movies-and-critic-reviews-dataset
Sourced two datasets:
1. rotten_tomatoes_critic_reviews.csv
2. rotten_tomatoes_movies.csv

In [4]:
review_df = pd.read_csv('rotten_tomatoes_critic_reviews.csv')
review_df = review_df[['rotten_tomatoes_link','review_content']]

review_df = review_df.dropna()
# for movies with multiple reviews, condense them all into a singleline of text (one massive concatenated review)
review_df_condensed = review_df.groupby('rotten_tomatoes_link', as_index=False)['review_content'].agg(' '.join)

In [5]:
print(review_df_condensed.head(5))
print('--------------------')
# First "concatenated" review of reviews
print(review_df_condensed.loc[0, 'review_content'])


              rotten_tomatoes_link  \
0                     m/+_one_2019   
1                             m/+h   
2                          m/-_man   
3  m/-cule_valley_of_the_lost_ants   
4                        m/0814255   

                                      review_content  
0  More than gets by on the strength of its funda...  
1  Ultimately Plush exceeds its very limited expe...  
2  Too many pieces of the puzzle are left out in ...  
3  A dialogue-free bug saga carried along by bril...  
4  A fantasy adventure that fuses Greek mythology...  
--------------------
More than gets by on the strength of its fundamentals: leads with crackling chemistry, crisper-than-average dialogue and a pair of directors who know how to stay out of the way. Erskine has a force of personality - and whip-crack comedic timing - that energizes this superior brand of amorous adventure. This charming rom com goes to a lot of expected places, though the appeal of the cast smoothes the ride greatly. Wit

In [6]:
movie_df = pd.read_csv('rotten_tomatoes_movies.csv') # other csv contains movie titles
movie_df = movie_df[['rotten_tomatoes_link','movie_title']]
movie_df_unique = movie_df.drop_duplicates(subset="rotten_tomatoes_link")

movie_and_reviews_df = review_df_condensed.merge(movie_df_unique, on="rotten_tomatoes_link", how="inner")
movie_and_reviews_df = movie_and_reviews_df[["movie_title", "review_content"]]


In [7]:
print(movie_and_reviews_df.head(10))

print('--------------------')
print(movie_and_reviews_df.loc[0, 'review_content'])

                                         movie_title  \
0  Percy Jackson & the Olympians: The Lightning T...   
1                                        Please Give   
2                                                 10   
3                    12 Angry Men (Twelve Angry Men)   
4                       20,000 Leagues Under The Sea   
5                                        10,000 B.C.   
6                                       The 39 Steps   
7                                       3:10 to Yuma   
8                          Charly (A Heartbeat Away)   
9                                    Abraham Lincoln   

                                      review_content  
0  A fantasy adventure that fuses Greek mythology...  
1  Like Holofcener's previous pictures, Please Gi...  
2  10 (1979) is known for its numerical rating sy...  
3  A film with texture, humour and relevance at a...  
4  [The] embodiment of Disney at his best -- fami...  
5  The mammoths, the savage, beaked jungle beasts... 

In [8]:
oscar_df = pd.read_csv('../ML_dataset.csv') # this csv contains 0/1 whether film was nominated for best pic
oscar_df = oscar_df[['title','best_picture_nom']]
oscar_df_unique = oscar_df.drop_duplicates(subset="title")


movie_and_reviews_df.rename(columns={'movie_title': 'title'}, inplace=True)

master_df = movie_and_reviews_df.merge(oscar_df_unique, on="title", how="inner")
master_df["review_content"] =  master_df["review_content"].str.lower() # convert everything to lowercase for glove embedding

In [9]:
print(master_df.head(50))
master_df.to_csv("master_df.tsv", sep="\t", index=False)

                                                title  \
0   Percy Jackson & the Olympians: The Lightning T...   
1                                         Please Give   
2                                                  10   
3                                        The 39 Steps   
4                                        3:10 to Yuma   
5                                     Abraham Lincoln   
6                                          Dark Water   
7                                         The Accused   
8                                       The Lost City   
9                                  The Breaking Point   
10                                         Adam's Rib   
11                         The Bridge of San Luis Rey   
12                                           Criminal   
13                       The Adventures of Mark Twain   
14                                          Deep Blue   
15                       The Adventures of Robin Hood   
16                             

In [10]:
# format an np array of tuples (label, concatenated review)
master_df = master_df.sample(frac=1, random_state=42).reset_index(drop=True) # shuffle before doing embeddings

master_tuples = list(
    master_df[["best_picture_nom", "review_content"]]
    .itertuples(index=False, name=None)
)
print(master_tuples[:5])

[(0, "both kidder and reeve shine. perhaps not as good as the first superman, this one offers more action and laughs a minute. the rare sequel that's even better than its predecessor. substantial credit should also go to terence stamp's casually menacing general zod, who transcends his terrible costume and make-up to appear a convincing threat. the controversy surrounding it continues to be much ado about nothing. it's a terrific follow-up to the incomparable original... no matter whose name is on the credits. almost as fantastic as the original. better than the first, but still overrated. a less inspired but mostly entertaining sequel. superman's battle with the three kryptonian villains is hammy but fun. relationship with lois lane is promptly abandoned. worthy follow up three kryptonian villains and superman kicking the snot out of each other and tearing up the town? who can't love this? the best superman. great villains and great clark insecurity subplot. superman's sequel is proba

### Feature Engineering  
Encode the concatenated movie reviews using glove embeddings

In [11]:
def load_feature_dictionary(file):
    """
    Creates a map of words to vectors using the file that has the glove
    embeddings.

    Parameters:
        file (str): File path to the glove embedding file.

    Returns:
        A dictionary indexed by words, returning the corresponding glove
        embedding np.ndarray.
    """
    glove_map = dict()
    with open(file, encoding='utf-8') as f:
        read_file = csv.reader(f, delimiter='\t')
        for row in read_file:
            word, embedding = row[0], row[1:]
            glove_map[word] = np.array(embedding, dtype=float)
    return glove_map

In [12]:
feature_dict = load_feature_dictionary("glove_embeddings.txt") # using glove embeddings
glove_get = feature_dict.get
rows = []

for label, review in master_tuples:
    vec = np.zeros(300)
    count = 0

    for word in review.split():
        wvec = glove_get(word)
        if wvec is not None:
            vec += wvec
            count += 1

    if count > 0:
        vec /= count  # find the average glove embedding

    rows.append([label, *vec])
    
output = np.array(rows)

np.savetxt(
    "formatted_glove_reviews.tsv",
    output,
    delimiter="\t",
    fmt="%.6f"
)

In [13]:
df = pd.read_csv("formatted_glove_reviews.tsv", sep="\t", header=None)
n = len(df)


# split data into training, testing, and validation
# 90%, 10%, 10% respectively
train_end = int(0.8 * n)
val_end = int(0.9 * n)
train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]
train_df.to_csv("train.tsv", sep="\t", index=False, header=False)
val_df.to_csv("val.tsv", sep="\t", index=False, header=False)
test_df.to_csv("test.tsv", sep="\t", index=False, header=False)